#  Prepare dataset from Kaggle

##  import used library

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import json

np.set_printoptions(
    precision=4,  # 4 chữ số thập phân
    suppress=True,  # không dùng dạng 1.23e-05
    linewidth=200,  # tránh xuống dòng quá sớm
)

In [ ]:
import os
import kagglehub
import shutil
from sklearn.feature_extraction.text import CountVectorizer
import sys

# Change to root of working dir
project_root = os.path.abspath("..")   # lên 1 cấp: MIND-research

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
path = kagglehub.dataset_download("arashnic/mind-news-dataset")

print("Dataset downloaded to: ", path)

Dataset downloaded to:  C:\Users\tonmi\.cache\kagglehub\datasets\arashnic\mind-news-dataset\versions\2


## Copy dataset to working folder

In [5]:
source = os.path.join(path, "MINDsmall_train")

destination = r"..\data\raw"

os.makedirs(destination, exist_ok=True)

files = [
    "news.tsv",
    "behaviors.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
]

for file in files:
    shutil.copy2(os.path.join(source, file), os.path.join(destination, file))

print("Done")

Done


##  Create sample dataset

- Sample dataset is used for dev and test model with sub dataset

### create behaviours dataset with n=10 sample

In [8]:
behaviours_path = os.path.join(destination, "behaviors.tsv")

columns_behaviours = ["user_id", "time", "history", "impressions"]

behaviours = pd.read_csv(
    behaviours_path,
    sep="\t",
    names=columns_behaviours,
)

behaviours = behaviours[["user_id", "history", "impressions"]]
behaviours = behaviours.dropna(subset=["history"])

sample_behaviours = behaviours.sample(n=10, random_state=42).reset_index(drop=True)

output_dir = r"..\data\sample"

sample_behaviours.to_csv(os.path.join(output_dir, "behaviours.csv"), index=False)

print(sample_behaviours.head())

  user_id                                            history  \
0  U10339          N31739 N12411 N11346 N61388 N12676 N15676   
1  U81911  N16082 N38457 N39481 N14734 N21241 N54659 N464...   
2  U77463                                             N22345   
3  U93135  N26136 N16233 N46978 N32483 N39117 N4020 N3399...   
4  U27678  N25691 N63842 N55388 N50155 N47558 N36920 N235...   

                                         impressions  
0  N16920-1 N14184-0 N62395-0 N453-0 N51187-0 N36...  
1                 N52122-0 N62360-0 N41020-0 N7319-1  
2  N1952-0 N39949-0 N62318-0 N36226-0 N17115-0 N3...  
3  N49180-0 N63970-0 N62360-0 N41020-0 N32544-0 N...  
4  N32759-0 N50058-0 N26262-0 N4156-0 N39432-0 N5...  


- **Sample dataset of news have to include all news that read by users**

In [9]:
user_behaviours = sample_behaviours.set_index("user_id")["history"].to_dict()
all_readed_news = set()

for key, value in user_behaviours.items():
    [all_readed_news.add(i) for i in value.split()]

print(len(all_readed_news))

346


- **Create news dataset with all news**

In [10]:
news_path = os.path.join(destination, "news.tsv")

if not os.path.exists(news_path):
    f_news_small = open(news_path, "x", encoding="utf-8")


columns = [
    "News_ID",
    "Category",
    "SubCategory",
    "Title",
    "Abstract",
    "URL",
    "Title_Entities",
    "Abstract_Entities",
]

news = pd.read_csv(
    news_path,
    sep="\t",
    names=columns,
)

news = news[["News_ID", "Category", "Title"]]

sample_news = news[news["News_ID"].isin(all_readed_news)].reset_index(drop=True)

output_dir = r"..\data\sample"

news.to_csv(os.path.join(output_dir, "news.csv"), index=False)

print(news.shape)
print(news.head())

(51282, 3)
  News_ID   Category                                              Title
0  N55528  lifestyle  The Brands Queen Elizabeth, Prince Charles, an...
1  N19639     health                      50 Worst Habits For Belly Fat
2  N61837       news  The Cost of Trump's Aid Freeze in the Trenches...
3  N53526     health  I Was An NBA Wife. Here's How It Affected My M...
4  N38324     health  How to Get Rid of Skin Tags, According to a De...


#   Attention

In sample dataset is used in develop model phrase, in this project sample dataset is not used since it is complete flow. 